# 10 — Deep-Learning Data Preparation

Generates Mel, MFCC, and chroma channels from the saved Notebook 04 4 kHz cycle WAVs, without repeating cycle extraction/resampling. It consumes the patient split created in Notebook 06. Features are cached in a compressed archive; augmentation is intentionally deferred to training loaders and will never touch validation/test cycles.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import numpy as np, pandas as pd
from src.post_05.common import FEATURES_DIR, ensure_output_dirs, new_artifact_path, PROCESSED
from src.post_05.cnn_utils import spectrogram_channels
ensure_output_dirs()
meta=pd.read_csv(PROCESSED/'processed_cycles.csv')
meta=meta[meta.processing_status.eq('success')].copy()
splits=pd.read_csv(FEATURES_DIR/'patient_splits.csv')
meta=meta.merge(splits[['patient_id','split']],on='patient_id',validate='many_to_one')
archive=FEATURES_DIR/'spectrogram_channels.npz'
if archive.exists():
    raise FileExistsError(f'Refusing to overwrite {archive}; use another cache filename explicitly.')
arrays=[]; failures=[]
for row in meta.itertuples(index=False):
    path=Path(str(row.processed_path)); path=path if path.is_absolute() else PROCESSED/path
    try: arrays.append(spectrogram_channels(path,sr=int(row.target_sample_rate)).astype(np.float16))
    except Exception as exc: failures.append({'patient_id':row.patient_id,'recording_id':row.recording_id,'cycle_number':row.cycle_number,'error':str(exc)}); arrays.append(None)
valid=np.array([a is not None for a in arrays])
meta=meta.loc[valid].reset_index(drop=True); x=np.stack([a for a in arrays if a is not None])
np.savez_compressed(new_artifact_path(archive),X=x,patient_id=meta.patient_id.to_numpy(),diagnosis=meta.diagnosis.astype(str).to_numpy(),recording_id=meta.recording_id.astype(str).to_numpy(),cycle_number=meta.cycle_number.to_numpy(),split=meta.split.astype(str).to_numpy(),processed_path=meta.processed_path.astype(str).to_numpy())
print('Saved channels:',x.shape,'dtype:',x.dtype,'modalities: mel, MFCC, chroma')
if failures:
    failure_path=new_artifact_path(FEATURES_DIR/'spectrogram_failures.csv'); pd.DataFrame(failures).to_csv(failure_path,index=False); print('Failed cycles:',len(failures))
print(meta.groupby(['split','diagnosis']).patient_id.nunique().unstack(fill_value=0))